<a href="https://colab.research.google.com/github/Biilzaa/TugasPraktikumDeployAI/blob/main/apk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
!pip install streamlit pyngrok --quiet

In [62]:
%%writefile apps.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


@st.cache_data
def prepare_model():

    df = pd.read_csv('Social_Network_Ads.csv')
    X = df[['Age', 'EstimatedSalary']]
    y = df['Purchased']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )

    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    return df, model, scaler, acc

def main():
    st.set_page_config(page_title="K-NN Ads Classifier", layout="centered", page_icon="👤")

    st.title("👥 Klasifikasi Target Iklan (K-NN)")
    st.markdown("---")

    try:

        df, model, scaler, acc = prepare_model()


        st.sidebar.header("Input Data Pelanggan")
        age = st.sidebar.slider("Umur (Tahun)", int(df.Age.min()), int(df.Age.max()), 30)
        salary = st.sidebar.slider("Estimasi Gaji ($)", int(df.EstimatedSalary.min()), int(df.EstimatedSalary.max()), 50000)


        st.subheader("Parameter Input")
        col1, col2 = st.columns(2)
        col1.metric("Umur Terpilih", f"{age} Thn")
        col2.metric("Gaji Terpilih", f"${salary:,}")

        if st.button("Prediksi Potensi Pembelian"):

            user_input = np.array([[age, salary]])
            user_input_scaled = scaler.transform(user_input)

            prediction = model.predict(user_input_scaled)[0]

            st.markdown("---")
            if prediction == 1:
                st.success("###Hasil: Pelanggan BERPOTENSI Membeli")
                st.balloons()
            else:
                st.warning("###Hasil: Pelanggan Cenderung TIDAK Membeli")

            st.info(f"Tingkat Akurasi Model: {acc*100:.2f}%")


        if st.checkbox("Lihat 5 Data Teratas"):
            st.write(df.head())

    except FileNotFoundError:
        st.error("Error: File 'Social_Network_Ads.csv' tidak ditemukan. Pastikan file sudah di-upload ke folder Colab.")

if __name__ == "__main__":
    main()

Overwriting apps.py


In [63]:
!streamlit run apps.py &>/dev/null&

In [64]:
from pyngrok import ngrok


NGROK_AUTH_TOKEN = "3DHk56b7kb6wJok3QtZQeXgUkCG_6PvipEKGM9NTvVEUbyvLR"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


ngrok.kill()

try:
    public_url = ngrok.connect(8501)
    print("Aplikasi Berhasil Dihubungkan!")
    print(f"Buka Link Ini: {public_url.public_url}")
except Exception as e:
    print(f"Koneksi gagal: {e}")

Aplikasi Berhasil Dihubungkan!
Buka Link Ini: https://guidable-unblessed-sports.ngrok-free.dev
